# "THE PRICE IS RIGHT" — Week 8, Day 4

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlanningAgent  
Day 5: The Price Is Right Finale

Today we give an OpenAI model three tools and let it coordinate the complete deal-finding workflow. We begin with fake functions so the tool-calling loop is easy to inspect, then switch to the real Week 8 agents.

In [1]:
import json
import logging
import os
from pathlib import Path

import chromadb
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find the repository .env file")
load_dotenv(dotenv_path, override=True)

REPO_ROOT = Path(dotenv_path).parent
WEEK8_DIR = REPO_ROOT / "lectures" / "week-eight"
MODEL = os.getenv("PLANNING_MODEL", "gpt-5-nano")
openai = OpenAI()

from agents.scanner_agent import ScannerAgent

logging.getLogger().setLevel(logging.INFO)
print(f"Planner model: {MODEL}")

Planner model: gpt-5-nano


/Users/marcolerma/GitHub/applied-llm-engineering/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Start with deterministic test data

`test_scan()` avoids live RSS and API calls while we learn the orchestration pattern.

In [2]:
test_results = ScannerAgent().test_scan()
test_results

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready


DealSelection(deals=[Deal(product_description="The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.", price=178.0, url='https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142'), Deal(product_description='The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and

## Create three pretend tools

These functions have the same signatures as the real agents, but their behavior is predictable and safe.

In [3]:
def scan_the_internet_for_bargains() -> str:
    """Return a fixed collection of example deals."""
    print("Fake scanner: returning the test deals")
    return test_results.model_dump_json()


def estimate_true_value(description: str) -> str:
    """Return a fixed estimate for one product."""
    print(f"Fake estimator: pricing {description[:40]}...")
    return json.dumps({"description": description, "estimated_true_value": 300})


def notify_user_of_deal(
    description: str,
    deal_price: float,
    estimated_true_value: float,
    url: str,
) -> str:
    """Simulate notifying the user without sending anything."""
    print(
        f"Fake notification: {description[:40]}... costs ${deal_price:.2f}; "
        f"estimated value ${estimated_true_value:.2f}; {url}"
    )
    return "Notification simulated"

In [4]:
notify_user_of_deal("A new iPhone", 100, 1000, "https://www.apple.com/iphone")

Fake notification: A new iPhone... costs $100.00; estimated value $1000.00; https://www.apple.com/iphone


'Notification simulated'

## Describe the functions to the model

A tool schema tells the model each function's name, purpose, arguments, and required fields. The application—not the model—executes the selected function.

In [5]:
scan_function = {
    "name": "scan_the_internet_for_bargains",
    "description": "Return example bargains and their advertised prices.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
}

estimate_function = {
    "name": "estimate_true_value",
    "description": "Estimate the true value of one product from its description.",
    "parameters": {
        "type": "object",
        "properties": {"description": {"type": "string"}},
        "required": ["description"],
        "additionalProperties": False,
    },
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Notify the user about the single best deal. Call at most once.",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {"type": "string"},
            "deal_price": {"type": "number"},
            "estimated_true_value": {"type": "number"},
            "url": {"type": "string"},
        },
        "required": ["description", "deal_price", "estimated_true_value", "url"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": scan_function},
    {"type": "function", "function": estimate_function},
    {"type": "function", "function": notify_function},
]
tools

[{'type': 'function',
  'function': {'name': 'scan_the_internet_for_bargains',
   'description': 'Return example bargains and their advertised prices.',
   'parameters': {'type': 'object',
    'properties': {},
    'required': [],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'estimate_true_value',
   'description': 'Estimate the true value of one product from its description.',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string'}},
    'required': ['description'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'notify_user_of_deal',
   'description': 'Notify the user about the single best deal. Call at most once.',
   'parameters': {'type': 'object',
    'properties': {'description': {'type': 'string'},
     'deal_price': {'type': 'number'},
     'estimated_true_value': {'type': 'number'},
     'url': {'type': 'string'}},
    'required': ['description', 'deal_price', 'estimated

In [6]:
def handle_tool_calls(message):
    """Execute every tool call requested in one assistant message."""
    mapping = {
        "scan_the_internet_for_bargains": scan_the_internet_for_bargains,
        "estimate_true_value": estimate_true_value,
        "notify_user_of_deal": notify_user_of_deal,
    }
    results = []
    for tool_call in message.tool_calls or []:
        name = tool_call.function.name
        function = mapping.get(name)
        arguments = json.loads(tool_call.function.arguments or "{}")
        result = function(**arguments) if function else f"Unknown tool: {name}"
        results.append({
            "role": "tool",
            "content": str(result),
            "tool_call_id": tool_call.id,
        })
    return results

In [7]:
system_message = (
    "You find bargains with your tools and notify the user about the single best one. "
    "Never invent products, prices, estimates, or URLs."
)
user_message = (
    "Scan for deals, estimate every deal's true value, choose the largest positive "
    "discount, notify the user exactly once, and finally reply OK."
)
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]

## Run the teaching loop

This cell makes paid OpenAI API calls. It uses only fake local tools and sends no real notification. The turn limit prevents an accidental infinite loop.

In [8]:
for turn in range(20):
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    message = response.choices[0].message
    messages.append(message)
    if not message.tool_calls:
        print(message.content)
        break
    messages.extend(handle_tool_calls(message))
else:
    raise RuntimeError("The demonstration exceeded its 20-turn safety limit")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Fake scanner: returning the test deals


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Fake estimator: pricing The Hisense R6 Series 55R6030N is a 55-i...
Fake estimator: pricing The Poly Studio P21 is a 21.5-inch LED p...
Fake estimator: pricing The Lenovo IdeaPad Slim 5 laptop is powe...
Fake estimator: pricing The Dell G15 gaming laptop is equipped w...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Fake notification: Hisense R6 Series 55R6030N 55" 4K UHD Ro... costs $178.00; estimated value $300.00; https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


OK


## Switch to the real Autonomous Planning Agent

The real planner uses:

1. `ScannerAgent` to fetch and select current deals.
2. `EnsembleAgent` to estimate each product's value using RAG, Modal, and the local neural network.
3. `MessagingAgent` to compose an alert and optionally deliver it through Pushover.

This requires the Day 1–3 setup, the Chroma collection, local model weights, Modal authentication/service, and an OpenAI API key. If Pushover variables are absent, delivery is safely skipped.

In [10]:
DB = str(WEEK8_DIR / "products_vectorstore")
client = chromadb.PersistentClient(path=DB)

# Day 2 writes to products_lite when LITE_MODE=True and products for the full dataset.
available = {item.name: item.count() for item in client.list_collections()}
collection_name = next(
    (name for name in ("products", "products_lite") if available.get(name, 0) > 0),
    None,
)
if collection_name is None:
    raise RuntimeError(
        f"No populated product collection found in {DB}. "
        "Run the Day 2 vector-store build cells first."
    )

collection = client.get_collection(collection_name)
print(f"Using {collection_name!r} with {collection.count():,} products for RAG")

Using 'products_lite' with 20,000 products for RAG


In [11]:
from agents.autonomous_planning_agent import AutonomousPlanningAgent

agent = AutonomousPlanningAgent(collection)

INFO:root:[Autonomous Planning Agent] Autonomous Planning Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using mps
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready
INFO:root:[Messaging Agent] Messaging Agent is initializin

The next cell performs the complete live workflow and may incur OpenAI/Modal usage. It only sends a push notification when both `PUSHOVER_USER` and `PUSHOVER_TOKEN` are configured.

In [12]:
opportunity = agent.plan()
opportunity

INFO:root:[Autonomous Planning Agent] Starting an autonomous planning run
INFO:root:[Autonomous Planning Agent] Calling Scanner Agent
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI
INFO:root:[Autonomous Planning Agent] Calling Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
12:07:25 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai
12:07:41 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using openai/gpt-5-nano
INFO:root:

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $199.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $84.51
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $198.34
INFO:root:[Autonomous Planning Agent] Calling Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
12:08:25 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai
12:08:38 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using open

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $1099.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $111.00
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $940.30
INFO:root:[Autonomous Planning Agent] Calling Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
12:08:49 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai
12:09:08 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using op

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $499.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $223.22
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $472.31
INFO:root:[Autonomous Planning Agent] Calling Ensemble Agent
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
12:09:39 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai
12:09:54 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using ope

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5-nano with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $499.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $256.13
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $455.50
INFO:root:[Autonomous Planning Agent] Calling Messaging Agent
INFO:root:[Messaging Agent] Messaging Agent is using openai/gpt-5-nano to craft the message
12:10:04 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai
12:10:10 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messa

Opportunity(deal=Deal(product_description='Refurbished EcoFlow River 3 Plus (286Wh) with 5000mAh power bank, Certified Refurbished, includes two-year warranty. Great portable power option for camping/backups.', price=159.0, url='https://www.dealnews.com/Refurb-EcoFlow-River-3-Plus-286-Wh-Powerstation-w-5-000-m-Ah-Power-Bank-for-159-free-shipping/22058938.html?iref=rss-c142'), estimate=198.34259967041018, discount=39.34259967041018)